# Heatwave & Mortality Descriptive Analysis

Analysis on `mortality_weather_weekly_labeled` (grain: NUTS2 region x week).

Shared calculation logic (direct standardization, relative risk, seasonal
restriction) lives in `src/analysis/epi_metrics.py`, imported below - not
redefined here - so the same functions can be reused by the Streamlit
dashboard without duplicating the formulas.

Notebook structure:
1. Data loading and sanity checks
2. Weekly mortality trend over time
3. Crude mortality rate
4. *(next sections: age standardization, heatwave RR, confounding)*


## 1. Data loading 

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import matplotlib.pyplot as plt

from src.analysis.epi_metrics import (
    ESP2013_WEIGHTS,
    age_specific_rates,
    direct_standardize,
    relative_risk,
    restrict_to_warm_season,
    linear_fit,
)
from src.utils.age_bins_loader import load_config as load_age_bins_config, get_bin_labels

pd.set_option('display.max_columns', 30)

df = pd.read_parquet("../data/analytics/views/mortality_weather_weekly_labeled.parquet")

print("Rows:", len(df))
print("Regions:", df['geo'].nunique())
print("Time range:", df['week_start_date'].min(), "-", df['week_start_date'].max())
df.head(3)

In [ ]:
# How many rows have missing data? We don't drop them here just check the extent,
# and decide case by case in each analysis whether and how to exclude them.
print("mortality_is_missing:", df['mortality_is_missing'].sum(), "/", len(df))
print("population_is_missing:", df['population_is_missing'].sum(), "/", len(df))

## 2. Weekly mortality trend over time

Sum of deaths (raw count, not average of rates) by macrozone, over time.
We sum raw deaths before dividing by population - never average already-computed rates,
to avoid giving small regions more weight than they deserve.

In [ ]:
# Aggregation by country + macrozone + week (sum of raw deaths)
weekly_by_macrozone = (
    df.groupby(['country', 'macrozone', 'week_start_date'], as_index=False)
    .agg(deaths=('deaths', 'sum'))
)

fig, ax = plt.subplots(figsize=(14, 6))
for (country, macrozone), group in weekly_by_macrozone.groupby(['country', 'macrozone']):
    group = group.sort_values('week_start_date')
    ax.plot(group['week_start_date'], group['deaths'], label=f"{country}-{macrozone}", alpha=0.7, linewidth=0.8)

ax.set_xlabel("Week")
ax.set_ylabel("Total deaths (sum across regions in the macrozone)")
ax.set_title("Weekly mortality trend by macrozone")
ax.legend(ncol=3, fontsize=8, loc='upper left')
plt.tight_layout()
plt.show()

**Reading note**: the regular, closely-spaced oscillations are normal seasonality
(more deaths in winter, typical of all European populations) - they should not be confused
with the effect of heatwaves, which is a summer phenomenon and more localized in time.
The two will be easier to tell apart once we explicitly compare weeks with
`any_heatwave_week=True` (next section).

## 3. Crude mortality rate

`rate = deaths / population x 100,000`

This must be computed **after** summing numerator and denominator separately when aggregating
by group (macrozone/country) - never as an average of already-computed regional rates (Simpson's
paradox: small regions would end up weighted the same as large ones).

In [ ]:
# We exclude rows with missing population ONLY for this specific calculation
# (the base dataset stays intact - the filter is local to this analysis)
valid = df[~df['population_is_missing']].copy()
print(f"Rows excluded for missing population: {(df['population_is_missing']).sum()} / {len(df)}")

# Crude rate for a single region x week
valid['crude_rate_per_100k'] = valid['deaths'] / valid['population'] * 100_000

# Crude rate aggregated by region (time average, only for a synthetic region-vs-region comparison)
rate_by_region = (
    valid.groupby(['geo', 'region_name', 'country', 'macrozone'], as_index=False)
    .agg(deaths=('deaths', 'sum'), population=('population', 'mean'))
)
rate_by_region['crude_rate_per_100k'] = rate_by_region['deaths'] / rate_by_region['population'] * 100_000 / valid['year'].nunique()

rate_by_region.sort_values('crude_rate_per_100k', ascending=False).head(10)

**Methodological note on the cell above**: here `population` is averaged over time (a convenient
proxy for a synthetic region-vs-region comparison across multiple years), and the rate is then divided
by the number of years to get an average annual rate. For a rigorous year by year comparison (e.g. for
the standardization in the next step), we'll work year by year explicitly instead of on a multi-year average.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 12))
plot_data = rate_by_region.sort_values('crude_rate_per_100k')
ax.barh(plot_data['region_name'] + " (" + plot_data['country'] + ")", plot_data['crude_rate_per_100k'])
ax.set_xlabel("Average annual crude mortality rate (per 100,000 inhabitants)")
ax.set_title("Crude mortality rate by region, 2015-2025")
plt.tight_layout()
plt.show()

## 4. Age-standardized mortality rate (direct standardization)

Standard population: **European Standard Population 2013 (ESP2013)**, published by Eurostat
(source: *Revision of the European Standard Population*, Eurostat Task Force, 2013 - weights
verified against the R package `PHEindicatormethods`, which reports them explicitly).

The 19 original ESP2013 five-year age groups are here reconciled to the 18 groups used in our
dataset (`Y_GE85` = sum of the two oldest groups `85-89` + `90+`), the same reconciliation logic
already applied between mortality_by_age and population_by_age at transform stage.

The weights themselves, plus the `age_specific_rates()` and `direct_standardize()` functions,
come from `src/analysis/epi_metrics.py` (imported above) rather than being redefined here.

In [ ]:
# ESP2013 weights imported from src/analysis/epi_metrics.py (ESP2013_WEIGHTS)
# Original order: 0-4,5-9,...,80-84,85-89,90+ (19 groups, sum 100,000, reconciled to 18)
print("ESP2013 weights loaded, sum:", sum(ESP2013_WEIGHTS.values()))
ESP2013_WEIGHTS

In [ ]:
df_age = pd.read_parquet("../data/analytics/views/mortality_by_age_weekly_labeled.parquet")
print("Rows:", len(df_age))
print("Age groups:", sorted(df_age['age'].unique()))

# Multi-year aggregation by region x sex x age (more stable denominators,
# as discussed: we avoid standardizing week by week on narrow age groups).
# Filtering stays here (analysis-specific decision), the aggregation itself
# is in age_specific_rates() (shared with the dashboard).
valid_age = df_age[~df_age['population_is_missing']].copy()

agg_age = age_specific_rates(valid_age, group_cols=['geo', 'region_name'])
agg_age.head(3)

In [ ]:
# Direct standardization via the shared function (src/analysis/epi_metrics.py)
standardized = direct_standardize(agg_age, ESP2013_WEIGHTS, group_cols=['geo', 'region_name'])

# Comparison with the crude rate computed earlier (section 3)
comparison = standardized.merge(
    rate_by_region[['geo', 'crude_rate_per_100k']], on='geo'
).sort_values('standardized_rate_per_100k', ascending=False)

comparison.head(10)

In [ ]:
# The interesting comparison: does the RANKING change between crude and standardized?
comparison['rank_crude'] = comparison['crude_rate_per_100k'].rank(ascending=False)
comparison['rank_standardized'] = comparison['standardized_rate_per_100k'].rank(ascending=False)
comparison['rank_change'] = comparison['rank_crude'] - comparison['rank_standardized']

print("Regions with the largest ranking shift after standardization:")
comparison.reindex(comparison['rank_change'].abs().sort_values(ascending=False).index)[
    ['region_name', 'crude_rate_per_100k', 'standardized_rate_per_100k', 'rank_crude', 'rank_standardized', 'rank_change']
].head(10)

**Reading note**: a region that climbs a lot in the ranking after standardization has a
younger-than-average population (its crude rate made it look "safer" than it really is, once age
structure is held constant). The opposite holds for a region that drops - exactly the phenomenon
described in the Ecuador/Sweden example from the course material.

## 5. Relative Risk (RR): weeks with vs without a heatwave

`RR = risk in the exposed / risk in the unexposed`, where "exposed" = weeks with
`any_heatwave_week=True`. Risk here is the mortality rate (deaths/population at risk over the
same period), consistent with the course's definition.

In [ ]:
exposed = valid[valid['any_heatwave_week'] == True]
unexposed = valid[valid['any_heatwave_week'] == False]

result_naive = relative_risk(
    exposed_deaths=exposed['deaths'].sum(),
    exposed_population=exposed['population'].sum(),
    unexposed_deaths=unexposed['deaths'].sum(),
    unexposed_population=unexposed['population'].sum(),
)
RR_naive = result_naive['rr']

print(f"Weeks with a heatwave: {len(exposed)}")
print(f"Weeks without a heatwave: {len(unexposed)}")
print(f"Risk (rate) in the exposed:   {result_naive['risk_exposed']*100_000:.2f} per 100,000 person-weeks")
print(f"Risk (rate) in the unexposed: {result_naive['risk_unexposed']*100_000:.2f} per 100,000 person-weeks")
print(f"RR (NOT corrected for season) = {RR_naive:.3f}")

### Seasonal confounding in the calculation above

The result (`RR < 1`) would seem to indicate that heat **reduces** mortality - an implausible
conclusion. The cause: weeks "with a heatwave" fall almost entirely between March and November,
while weeks "without" **also include the entire winter** (January, February, December), which has
the highest baseline mortality of the year for reasons entirely unrelated to heat (flu, winter
respiratory illness). The comparison group is therefore contaminated by a confounding factor -
season - which must be controlled for by restricting the comparison to the same seasonal window.

In [ ]:
# Correction: compare ONLY weeks within the same seasonal window
# (March-November, the period during which heatwaves can occur)
warm_season = restrict_to_warm_season(valid, date_col='week_start_date', start_month=3, end_month=11)

exposed_ws = warm_season[warm_season['any_heatwave_week'] == True]
unexposed_ws = warm_season[warm_season['any_heatwave_week'] == False]

result_corrected = relative_risk(
    exposed_deaths=exposed_ws['deaths'].sum(),
    exposed_population=exposed_ws['population'].sum(),
    unexposed_deaths=unexposed_ws['deaths'].sum(),
    unexposed_population=unexposed_ws['population'].sum(),
)
RR = result_corrected['rr']

print(f"Exposed weeks: {len(exposed_ws)}")
print(f"Unexposed weeks (same seasonal window, Mar-Nov): {len(unexposed_ws)}")
print(f"Risk in exposed:   {result_corrected['risk_exposed']*100_000:.2f} per 100,000")
print(f"Risk in unexposed: {result_corrected['risk_unexposed']*100_000:.2f} per 100,000")
print(f"RR (corrected for season) = {RR:.3f}")

**Interpretation**: after correction, `RR ≈ 1.01` - a positive but modest effect. Plausible
reasons for a small effect (to be stated as limitations in the report):
- The comparison is within the same week as exposure, with no lag (see the note on harvesting below)
- **All-cause** mortality includes many deaths unrelated to heat, which dilutes the signal
- Pooling across 59 regions and 11 years averages out localized extreme events that could have
  much higher RR at the single-region/year level (e.g. the 2022 European heatwave)

This naive-vs-corrected comparison is itself a direct demonstration of **confounding** (Section 6),
worth including explicitly in the report as a practical example alongside the standardization one.

**Limitation to state explicitly**: this comparison is within the same week as exposure.
The heat-mortality literature often shows a 1-2 week delayed effect ("harvesting"/mortality
displacement) - an RR computed without a lag likely **underestimates** the true effect. Not
implemented here (out of scope, consistent with excluding DLNM), but should be mentioned in the
report's limitations.

In [ ]:
# RR by country, to see whether the effect is homogeneous or varies geographically
rr_by_country = []
for country, group in warm_season.groupby('country'):
    exp = group[group['any_heatwave_week'] == True]
    nonexp = group[group['any_heatwave_week'] == False]
    if len(exp) == 0 or len(nonexp) == 0:
        continue
    r = relative_risk(
        exposed_deaths=exp['deaths'].sum(),
        exposed_population=exp['population'].sum(),
        unexposed_deaths=nonexp['deaths'].sum(),
        unexposed_population=nonexp['population'].sum(),
    )
    rr_by_country.append({'country': country, 'RR': r['rr'], 'n_exposed_weeks': len(exp)})

pd.DataFrame(rr_by_country).sort_values('RR', ascending=False)

### Simple linear regression: temperature vs mortality rate

In [ ]:
valid['crude_rate_per_100k'] = valid['deaths'] / valid['population'] * 100_000

fit = linear_fit(valid['temperature_2m_mean'], valid['crude_rate_per_100k'])

fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(valid['temperature_2m_mean'], valid['crude_rate_per_100k'], alpha=0.05, s=5)
x_line = np.linspace(valid['temperature_2m_mean'].min(), valid['temperature_2m_mean'].max(), 100)
ax.plot(x_line, fit['slope'] * x_line + fit['intercept'], color='red', linewidth=2,
        label=f"y = {fit['slope']:.3f}x + {fit['intercept']:.1f}  (r = {fit['r']:.3f})")
ax.set_xlabel("Weekly mean temperature (°C)")
ax.set_ylabel("Mortality rate (per 100,000)")
ax.set_title("Temperature-mortality relationship (all regions, all weeks)")
ax.legend()
plt.tight_layout()
plt.show()

**Reading note**: a linear regression assumes a monotonic relationship, but temperature-related
mortality is typically **U- or J-shaped** (higher risk at both extreme cold and extreme heat, minimum
at some intermediate "comfort" temperature). A single straight line across the whole temperature range
likely **flattens** both extremes. This is exactly the limitation that DLNM (out of scope here) would
address by explicitly modeling the non-linearity - the contrast between this simple baseline and a
hypothetical DLNM is a suggested talking point for the report.

## 6. Confounding: age structure

Empirical demonstration that age structure is a confounder in the comparison across regions:
- we compute the share of population aged 65+ per region
- we check the correlation with the crude mortality rate
- expected confounder: the older the population, the higher the crude rate, regardless of
  heat exposure - exactly why standardization (section 4) is necessary before comparing regions
  to each other.

In [ ]:
# Share of population aged 65+ per region (from the age dataset, averaged over the period)
over65_bins = ['Y65-69', 'Y70-74', 'Y75-79', 'Y80-84', 'Y_GE85']

pop_by_region_age = (
    valid_age.groupby(['geo', 'age'], as_index=False)
    .agg(population=('population', 'mean'))
)
pop_total = pop_by_region_age.groupby('geo')['population'].sum().rename('pop_total')
pop_over65 = pop_by_region_age[pop_by_region_age['age'].isin(over65_bins)].groupby('geo')['population'].sum().rename('pop_over65')

share_over65 = pd.concat([pop_total, pop_over65], axis=1)
share_over65['share_over65'] = share_over65['pop_over65'] / share_over65['pop_total']

confounding_df = comparison.merge(share_over65[['share_over65']], on='geo')

corr = confounding_df['share_over65'].corr(confounding_df['crude_rate_per_100k'])
print(f"Correlation between share of population 65+ and crude mortality rate: r = {corr:.3f}")

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(confounding_df['share_over65']*100, confounding_df['crude_rate_per_100k'])
ax.set_xlabel("Share of population aged 65+ (%)")
ax.set_ylabel("Crude mortality rate (per 100,000)")
ax.set_title(f"Age structure as a confounder (r={corr:.2f})")
plt.tight_layout()
plt.show()

**Conclusion**: the positive correlation confirms that regions with an older population show
higher crude rates independently of other factors - this is precisely the age-confounding mechanism
described in the course material (Ch. 3, Ecuador/Sweden example), here verified empirically across
the 59 regions in the dataset.

## 7. Effect modification by age: does the heatwave RR vary by age group?

Section 6 treated age as a **confounder** to adjust away (standardization). Here the question is
different: not "is the *comparison across regions* biased by age structure", but "does the *effect
of heat itself* differ across age groups" - i.e. **effect modification**, not confounding. The
epidemiological hypothesis (and the reason this dataset was built with age granularity at all) is
that older people are more physiologically vulnerable to heat stress.

This uses `mortality_by_age_weekly_stratified_heat` (built by `build_analysis_view.py` from the
`stratified_heat` scheme in `config/age_bins.yaml`): `under_65` / `65-74` / `75-84` / `85+`. Note
this is a **different** grouping purpose than the ESP2013 standardization in section 4 - that one
needs the fine-grained quinquennial bins to match the ESP2013 weight table; this one deliberately
uses a coarse reference band (`under_65`) since young-age heat risk isn't the research question,
against a finer split of the elderly range where the hypothesis is expected to show up.

In [ ]:
df_stratified = pd.read_parquet("../data/analytics/views/mortality_by_age_weekly_stratified_heat.parquet")
print("Rows:", len(df_stratified))
print("Age groups:", df_stratified['age_group'].unique())
print("Sexes:", df_stratified['sex'].unique())  # M/F only - no 'T' total, so summing both is correct, not double-counting

# Filter missing population, same principle as elsewhere: local to this analysis
valid_stratified = df_stratified[~df_stratified['population_is_missing']].copy()

# Collapse sex (M+F) - not the research question here, and safe to sum since
# there's no separate 'T' row that would double-count.
by_age_group = (
    valid_stratified.groupby(['geo', 'age_group', 'year', 'week', 'week_start_date', 'any_heatwave_week'], as_index=False)
    .agg(deaths=('deaths', 'sum'), population=('population', 'sum'))
)
by_age_group.head(3)

As in section 5, the comparison must be restricted to the same seasonal window
(March-November) before computing RR, to avoid the same seasonal confounding diagnosed there -
this restriction isn't specific to age, so it's applied here exactly the same way, via the shared
`restrict_to_warm_season()`.

In [ ]:
warm_season_by_age = restrict_to_warm_season(by_age_group, date_col='week_start_date', start_month=3, end_month=11)

# Preserve the age_group display order defined in config/age_bins.yaml (under_65, 65-74, 75-84, 85+),
# rather than relying on whatever order groupby happens to produce.
age_bins_config = load_age_bins_config()
age_group_order = get_bin_labels(age_bins_config, 'stratified_heat')

rr_by_age_group = []
for age_group in age_group_order:
    group = warm_season_by_age[warm_season_by_age['age_group'] == age_group]
    exp = group[group['any_heatwave_week'] == True]
    nonexp = group[group['any_heatwave_week'] == False]
    if len(exp) == 0 or len(nonexp) == 0:
        continue
    r = relative_risk(
        exposed_deaths=exp['deaths'].sum(),
        exposed_population=exp['population'].sum(),
        unexposed_deaths=nonexp['deaths'].sum(),
        unexposed_population=nonexp['population'].sum(),
    )
    rr_by_age_group.append({'age_group': age_group, 'RR': r['rr'],
                            'risk_exposed_per_100k': r['risk_exposed']*100_000,
                            'risk_unexposed_per_100k': r['risk_unexposed']*100_000,
                            'n_exposed_weeks': len(exp)})

rr_by_age_group_df = pd.DataFrame(rr_by_age_group)
rr_by_age_group_df

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(rr_by_age_group_df['age_group'], rr_by_age_group_df['RR'], color='firebrick')
ax.axhline(1.0, color='black', linewidth=1, linestyle='--', label='RR = 1 (no effect)')
ax.set_xlabel("Age group")
ax.set_ylabel("Relative risk (heatwave vs non-heatwave weeks)")
ax.set_title("Heatwave RR by age group (season-restricted, Mar-Nov)")
ax.legend()
plt.tight_layout()
plt.show()

**Reading note**: if the vulnerability hypothesis holds, RR should increase monotonically from
`under_65` to `85+`. A flat or non-monotonic pattern would suggest either that all-cause mortality
dilutes the heat-specific signal similarly across ages, or that the season restriction and lack of
lag (same limitations noted in section 5) are masking a real age gradient - both worth discussing
in the report rather than treated as a negative result on their own.

**Limitations carried over from section 5, equally applicable here**: no lag between exposure and
outcome (harvesting/mortality displacement may matter more for the oldest group, where deaths could
be brought forward by days rather than in the same week), and no further age standardization within
each group (each RR compares heatwave vs non-heatwave weeks *within* the same age band, so the age
band's own internal composition doesn't confound this specific comparison - but very wide bands like
`under_65` still average over very different baseline risks).